# Viking V5 Ultraviolet Auroral Imager — Implementation / 구현

**Paper**: Anger et al., "An Ultraviolet Auroral Imager for the Viking Spacecraft", *Geophysical Research Letters*, 14(4), 387–390, 1987. DOI: 10.1029/GL014i004p00387

**English** — This notebook reproduces three engineering models from the V5 paper:
1. **Photometric chain** — convert auroral column emission rate (rayleighs) to digital number per pixel-spin, including filter, photocathode, MCP gain, and electro-optic gain.
2. **Visible-light leakage budget (dayglow suppression model)** — multiply the suppression factors of filter, photocathode, MCP scatter, and aluminum overcoat to verify that sub-kR aurora is detectable on a sunlit Earth.
3. **Electronic despinning kinematics + image dewarping** — derive the vertical-clock frequency that matches spin-induced image motion, and demonstrate the column-tilt smearing tolerance (±0.2°) by simulation.

**한국어** — 본 노트북은 V5 논문의 세 가지 공학 모델을 재현한다.
1. **측광 체인** — 오로라 강도(레일리) → 픽셀-스핀당 디지털 신호 변환(필터, 광음극, MCP 이득, 전기-광학 이득 포함).
2. **가시광 누설 예산(dayglow 억제 모델)** — 필터, 광음극, MCP 산란, Al 오버코트의 억제 인자를 곱해 sunlit Earth 배경에서도 sub-kR 오로라 검출 가능성 검증.
3. **전자식 despinning 운동학 + 영상 dewarp** — 스핀에 의한 영상 이동에 맞는 수직 클록 주파수를 유도하고, 컬럼 정렬 오차에 따른 번짐(±0.2° 허용)을 시뮬레이션.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## Part 1: Photometric chain / 측광 체인

**English** — Compute photoelectrons per pixel-spin and digital signal in dn for an auroral column emission rate. Verify the paper's claim that sub-kilo-rayleigh aurora produces a few photoelectrons per pixel-spin.

**한국어** — 오로라 기둥 발광율(R)에 대해 픽셀-스핀당 광전자 수와 dn 신호를 계산하여, sub-kR 오로라가 픽셀-스핀당 수 광전자를 만든다는 논문 주장을 검증.

In [ ]:
@dataclass
class V5Camera:
    """Viking V5 imager radiometric model.

    Attributes follow the camera characteristics in Anger et al. (1987), Table 1.

    Attributes:
        focal_length_mm: Optical focal length in millimeters.
        f_number: Optical f-number (focal ratio).
        pixel_size_um: Effective square pixel size projected onto the MCP.
        filter_transmission: Peak in-band filter transmission (0-1).
        photocathode_qe: Photocathode quantum efficiency at the operating
            wavelength (electrons per incident photon).
        electro_optic_gain_dn_per_pe: Detector electro-optic gain in digital
            numbers per photoelectron (paper quotes 0.1-0.4).
        central_obscuration_frac: Fraction of pupil area lost to the
            secondary mirror obscuration.
    """

    focal_length_mm: float = 22.4
    f_number: float = 1.0
    pixel_size_um: float = 30.0
    filter_transmission: float = 0.4
    photocathode_qe: float = 0.10
    electro_optic_gain_dn_per_pe: float = 0.2
    central_obscuration_frac: float = 0.5

    @property
    def aperture_diameter_mm(self) -> float:
        """Return entrance aperture diameter D = f / N."""
        return self.focal_length_mm / self.f_number

    @property
    def effective_area_cm2(self) -> float:
        """Return effective collecting area in cm^2 after central obscuration."""
        d_cm = self.aperture_diameter_mm * 0.1
        a_geom = np.pi * (d_cm / 2.0) ** 2
        return a_geom * (1.0 - self.central_obscuration_frac)

    @property
    def pixel_solid_angle_sr(self) -> float:
        """Return pixel solid angle Omega = (p / f)^2 in steradians."""
        theta = (self.pixel_size_um * 1e-3) / self.focal_length_mm
        return theta ** 2

    @property
    def pixel_angular_resolution_deg(self) -> float:
        """Return per-pixel angular resolution in degrees."""
        theta = (self.pixel_size_um * 1e-3) / self.focal_length_mm
        return np.degrees(theta)

    def signal_dn(self, rayleighs: float, exposure_s: float = 1.0) -> dict:
        """Compute the per-pixel digital signal for a given auroral brightness.

        Args:
            rayleighs: Auroral column emission rate in rayleighs (1 R =
                1e6/(4*pi) photons cm^-2 s^-1 sr^-1).
            exposure_s: Effective exposure time in seconds.

        Returns:
            Dictionary with photon rate, photoelectron count, and digital
            number for one pixel-spin.
        """
        intensity = rayleighs * 1e6 / (4.0 * np.pi)
        photon_rate = (
            intensity
            * self.pixel_solid_angle_sr
            * self.effective_area_cm2
        )
        photoelectron_rate = (
            photon_rate * self.filter_transmission * self.photocathode_qe
        )
        n_pe = photoelectron_rate * exposure_s
        dn = n_pe * self.electro_optic_gain_dn_per_pe
        return {
            'photon_rate_per_s': photon_rate,
            'photoelectrons': n_pe,
            'dn': dn,
        }


# Camera 1 model: KBr photocathode at 1356 A.
cam1 = V5Camera(filter_transmission=0.4, photocathode_qe=0.10)
print(f'Aperture D       = {cam1.aperture_diameter_mm:.2f} mm')
print(f'A_eff            = {cam1.effective_area_cm2:.2f} cm^2')
print(f'Pixel resolution = {cam1.pixel_angular_resolution_deg:.4f} deg '
      f'(paper: 0.076 deg)')
print(f'Pixel Omega      = {cam1.pixel_solid_angle_sr:.2e} sr')

In [ ]:
# Sweep auroral brightness from 100 R to 30 kR (IBC I to IBC III+).
brightness_R = np.logspace(2, 4.5, 60)
results = [cam1.signal_dn(r, exposure_s=1.0) for r in brightness_R]
n_pe = np.array([r['photoelectrons'] for r in results])
dn = np.array([r['dn'] for r in results])

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.loglog(brightness_R, n_pe, 'b-', label='photoelectrons / pixel-spin')
ax1.set_xlabel('Auroral column emission rate [R]')
ax1.set_ylabel('Photoelectrons per pixel-spin', color='b')
ax1.axhline(3, color='b', linestyle='--', alpha=0.5,
            label='detection floor (3 pe)')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True, which='both', alpha=0.3)

ax2 = ax1.twinx()
ax2.loglog(brightness_R, dn, 'r-', label='dn / pixel-spin')
ax2.axhline(255, color='r', linestyle=':', alpha=0.5,
            label='8-bit saturation')
ax2.set_ylabel('Digital number (dn)', color='r')
ax2.tick_params(axis='y', labelcolor='r')

# Mark IBC reference levels.
for ibc, val in [('IBC I (1 kR)', 1e3), ('IBC II (10 kR)', 1e4)]:
    ax1.axvline(val, color='k', alpha=0.3, linestyle=':')
    ax1.text(val, 1.5, ibc, rotation=90, fontsize=10, alpha=0.6)

plt.title('Viking V5 Camera 1 photometric response (1356 A, 1 s exposure)')
fig.tight_layout()
plt.show()

# Spot check: 1 kR should give a few pe / pixel-spin.
kr = cam1.signal_dn(1000.0, exposure_s=1.0)
print(f"\n1 kR -> {kr['photoelectrons']:.1f} pe, {kr['dn']:.2f} dn")
print(f"100 R -> {cam1.signal_dn(100.0)['photoelectrons']:.2f} pe "
      f"(below floor; binning required)")

## Part 2: Visible-light leakage budget — dayglow suppression model / 가시광 누설 예산 — dayglow 억제 모델

**English** — V5 must image aurora against a sunlit Earth. Visible-light leakage is suppressed in four serial stages: filter, photocathode, MCP multiple-scattering, and a thin aluminum overcoat on the phosphor. We multiply the suppression factors and compare the residual visible signal from a sunlit Earth to an IBC II auroral signal.

**한국어** — V5는 sunlit Earth를 배경으로 오로라를 본다. 가시광 누설은 네 단계로 억제된다: 필터, 광음극, MCP 다중 산란, 형광체 위 Al 오버코트. 각 억제 인자를 곱해 IBC II 오로라 신호와 비교한다.

In [ ]:
def cathode_qe_visible(wavelength_a: np.ndarray) -> np.ndarray:
    """Approximate residual photocathode QE in the visible.

    Per Anger et al. (1987), the residual QE is < 2e-8 at 3000 A and falls
    by a factor of 10 for every 290 A increase in wavelength.

    Args:
        wavelength_a: Wavelength in Angstroms.

    Returns:
        Residual quantum efficiency (electrons per photon).
    """
    qe_3000 = 2.0e-8
    decades = (wavelength_a - 3000.0) / 290.0
    return qe_3000 * 10.0 ** (-decades)


def visible_leakage_chain(wavelength_a: float) -> dict:
    """Compute the multilayer visible-light suppression chain.

    Args:
        wavelength_a: Wavelength in Angstroms (2000-7000 typical).

    Returns:
        Dictionary of stage-by-stage suppression factors and the product.
    """
    # Bandpass-filter long-wavelength leakage (rough out-of-band level).
    filter_leak = 1.0e-4 if wavelength_a > 2000 else 1.0
    cathode_leak = float(cathode_qe_visible(np.array([wavelength_a]))[0])
    mcp_scatter_leak = 1.0 / 5.0e3  # ~5000x suppression per the paper.
    al_overcoat_leak = 1.0e-2       # ~200 A Al on phosphor blocks visible.
    total = filter_leak * cathode_leak * mcp_scatter_leak * al_overcoat_leak
    return {
        'filter': filter_leak,
        'photocathode': cathode_leak,
        'mcp_scatter': mcp_scatter_leak,
        'al_overcoat': al_overcoat_leak,
        'total': total,
    }


# Show stage-by-stage budget at the visible peak (~5500 A).
budget = visible_leakage_chain(5500.0)
for stage, factor in budget.items():
    print(f'  {stage:14s}: {factor:.2e}')

In [ ]:
# Compare residual visible signal from sunlit Earth vs. an IBC II aurora.
# Sunlit Earth top-of-atmosphere visible radiance ~ 1e10 photons cm^-2 s^-1
# sr^-1 (broadband ~5500 A); aurora at IBC II = 10 kR = 7.96e8 photons.
sunlit_earth_radiance = 1.0e10  # photons cm^-2 s^-1 sr^-1 (broadband)
ibc2_radiance = 1.0e4 * 1e6 / (4.0 * np.pi)

# Compute effective signals at the photocathode using the leakage chain.
wavelengths = np.array([3000.0, 3500.0, 4000.0, 4500.0, 5000.0, 5500.0,
                        6000.0, 6500.0])
residual_visible = []
for w in wavelengths:
    chain = visible_leakage_chain(w)
    residual_visible.append(sunlit_earth_radiance * chain['total'])
residual_visible = np.array(residual_visible)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(wavelengths, residual_visible, 'o-', label='residual visible '
            'after V5 chain')
ax.axhline(ibc2_radiance, color='r', linestyle='--',
           label='IBC II aurora signal (effective radiance)')
ax.set_xlabel('Visible wavelength [A]')
ax.set_ylabel('Effective radiance reaching detector '
              '[photons cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
ax.set_title('Dayglow suppression chain — V5 visible-light rejection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Margin in decibels: how many decades below IBC II the residual visible is.
ratio = ibc2_radiance / residual_visible[wavelengths == 5500][0]
print(f'\nIBC II aurora / residual visible at 5500 A = {ratio:.2e}')
print('(positive = aurora dominates; the V5 chain achieves > 1e10 margin '
      'against sunlit Earth)')

## Part 3: Electronic despinning kinematics + column-tilt smearing / 전자식 despinning 운동학과 컬럼 정렬 번짐

**English** — V5 achieves 1-s exposure on a spinning satellite by stepping CCD column charges in synchrony with image motion. We derive the required vertical-clock frequency and show by simulation that column tilts above ±0.2° produce visible smearing — exactly the alignment tolerance the paper specifies.

**한국어** — V5는 회전 위성에서 1초 노출을 얻기 위해 CCD 컬럼 전하를 영상 이동과 동기 이송한다. 필요한 수직 클록 주파수를 유도하고, 컬럼 정렬 오차 ±0.2° 초과 시 번짐이 발생함을 시뮬레이션으로 확인한다.

In [ ]:
def vertical_clock_frequency(
    spin_rpm: float,
    focal_length_mm: float = 22.4,
    pixel_pitch_um: float = 30.0,
) -> float:
    """Compute the despinning vertical-clock frequency.

    The image moves across the focal plane at v = f * omega; matching CCD
    charge motion requires a vertical clock at v / p rows per second.

    Args:
        spin_rpm: Spacecraft spin rate in revolutions per minute.
        focal_length_mm: Camera focal length in millimeters.
        pixel_pitch_um: CCD pixel pitch in micrometers.

    Returns:
        Vertical-clock frequency in rows per second.
    """
    omega = spin_rpm * 2.0 * np.pi / 60.0
    v_mm_s = focal_length_mm * omega
    return v_mm_s * 1000.0 / pixel_pitch_um


for rpm in [1.0, 3.0, 10.0]:
    f_vclk = vertical_clock_frequency(rpm)
    print(f'spin = {rpm:5.1f} rpm  -> vertical clock = {f_vclk:8.1f} rows/s  '
          f'(288 rows traverse in {288 / f_vclk:.2f} s)')

In [ ]:
def simulate_smearing(
    column_tilt_deg: float,
    n_rows: int = 288,
    n_cols: int = 385,
    star_pos: tuple = (192, 144),
) -> np.ndarray:
    """Simulate column-tilt smearing during electronic despinning.

    A point source is integrated over the row-by-row charge transfer; if the
    CCD columns are tilted relative to the spin direction by angle beta, the
    accumulated charge spreads laterally by N_rows * tan(beta) pixels.

    Args:
        column_tilt_deg: Column-to-spin-equator tilt angle in degrees.
        n_rows: CCD image-region row count.
        n_cols: CCD image-region column count.
        star_pos: (column, row) position of the simulated point source.

    Returns:
        2D image array of the smeared source.
    """
    image = np.zeros((n_rows, n_cols))
    col0, row0 = star_pos
    tan_beta = np.tan(np.radians(column_tilt_deg))
    for row_step in range(n_rows):
        # Each charge-transfer step shifts the source laterally by tan(beta)
        # pixels relative to the column axis.
        shift = row_step * tan_beta
        col = col0 + shift
        col_int = int(np.floor(col))
        col_frac = col - col_int
        row = row0 - row_step
        if 0 <= row < n_rows:
            if 0 <= col_int < n_cols - 1:
                image[row, col_int] += (1.0 - col_frac)
                image[row, col_int + 1] += col_frac
    return image


tilts = [0.0, 0.1, 0.2, 0.5, 1.0]
fig, axes = plt.subplots(1, len(tilts), figsize=(16, 4))
for ax, tilt in zip(axes, tilts):
    img = simulate_smearing(tilt)
    # Crop to a region around the source for visibility.
    crop = img[100:200, 170:220]
    ax.imshow(crop, cmap='gray_r', origin='lower', aspect='auto')
    ax.set_title(f'tilt = {tilt:.1f} deg')
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('Column-tilt smearing simulation — V5 despinning tolerance')
plt.tight_layout()
plt.show()

# Quantify smearing as integrated full-width.
for tilt in tilts:
    img = simulate_smearing(tilt)
    profile = img.sum(axis=0)
    nonzero = np.where(profile > 0.1 * profile.max())[0]
    width = nonzero.max() - nonzero.min() + 1 if len(nonzero) > 0 else 0
    print(f'tilt = {tilt:4.1f} deg  ->  smear width = {width:3d} pixels '
          f'(spec: <= 1 pixel)')

**English** — The simulation reproduces the paper's geometric requirement: tilt $\beta$ produces smear width $\Delta x \approx N_{\text{rows}}\tan\beta$. For $N=288$ and $\Delta x \leq 1$ pixel: $\tan\beta \leq 1/288 \approx 0.0035$, i.e. $\beta \leq 0.20^\circ$ — exactly the ±0.2° spec quoted in the paper.

**한국어** — 시뮬레이션이 논문의 기하 요구를 그대로 재현한다. 정렬 오차 $\beta$의 번짐 폭 $\Delta x \approx N_{\text{rows}}\tan\beta$. $N=288$에서 $\Delta x \leq 1$ 픽셀이려면 $\tan\beta \leq 1/288 \approx 0.0035$, 즉 $\beta \leq 0.20^\circ$ — 논문의 ±0.2° 사양과 정확히 일치.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Auroral UV imager | Viking V5 (1986), 2 cameras, curved MCP, fiber taper | IMAGE FUV (2000): WIC + SI13 + SI12, three dedicated channels |
| Photocathode | KBr / CsI deposited on curved MCP front | KBr/CsI on flat MCP with field-flattener fiber input |
| Detector | English Electric Valve P8602 frame-transfer CCD, 288×385 | Back-illuminated CCD or CMOS APS, 1024×1024+ |
| Spin compensation | Electronic despin via vertical-clock charge shift | Three-axis stabilized platforms or fast-readout dewarp |
| Visible suppression | Filter + photocathode + MCP scatter + Al overcoat (~10²² total) | Same multilayer chain, refined coatings (Acton, Astropak) |
| Telemetry | 5.1–30.7 kbps, ground-uploadable parameter lists | High-rate Ka-band, software-defined operations |

**English** — The V5 photometric chain (Part 1) is reproduced quantitatively: 1 kR aurora gives ~10 photoelectrons / pixel-spin and ~2 dn / pixel-spin, supporting the paper's sub-kR detectability claim. The dayglow suppression budget (Part 2) shows the photocathode as the dominant rejection stage, with the multilayer product giving > 10²² of visible-light rejection. The despinning kinematics (Part 3) recover the paper's ±0.2° column-alignment tolerance directly from the geometry $\Delta x = N_{\text{rows}}\tan\beta \leq 1$ pixel.

**한국어** — Part 1의 측광 체인 재현으로 1 kR 오로라가 픽셀-스핀당 ~10 광전자, ~2 dn임을 확인하여 논문의 sub-kR 검출 주장을 검증. Part 2의 dayglow 억제 예산은 광음극이 핵심 억제 단계이며 다층 곱이 > 10²² 가시광 거부를 제공함을 보였다. Part 3의 despinning 운동학은 ±0.2° 정렬 사양이 $\Delta x = N_{\text{rows}}\tan\beta \leq 1$ 픽셀이라는 기하에서 직접 도출됨을 재현.